In [4]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 4.0 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import pandas as pd
import re
import hashlib
from rapidfuzz import fuzz

# ========= NORMALISATION =========
def normalize(text):
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text.lower().strip())

def extract_doi(text):
    match = re.search(r'10\.\S+', text)
    return match.group(0) if match else ""

# ========= IEEE PARSER =========
def parse_ieee(text):
    records = []
    blocks = text.split("\n\n")

    for block in blocks:
        if "doi:" not in block:
            continue

        title_match = re.search(r'"(.+?)"', block)
        title = title_match.group(1) if title_match else ""
        authors_match = re.search(r'^(.*?),\s*"', block)
        authors = authors_match.group(1) if authors_match else ""

        doi = extract_doi(block)

        records.append({
            "Database": "IEEE",
            "Title": title,
            "DOI": doi,
            "PMID": "",
            "Authors": authors,
        })

    return records

# ========= SCOPUS PARSER =========
def parse_scopus(text):
    records = []
    entries = text.split("\n\n")

    for entry in entries:
        if "DOI:" not in entry:
            continue

        title_match = re.search(r'\n(.+?)\n\(', entry)
        title = title_match.group(1) if title_match else ""

        doi_match = re.search(r'DOI:\s*(10\.\S+)', entry)
        doi = doi_match.group(1) if doi_match else ""

        pmid_match = re.search(r'PUBMED ID:\s*(\d+)', entry)
        pmid = pmid_match.group(1) if pmid_match else ""
        authors_match = re.search(r'AUTHOR FULL NAMES:\s*(.+)', entry)
        authors = authors_match.group(1) if authors_match else ""

        records.append({
            "Database": "Scopus",
            "Title": title,
            "DOI": doi,
            "PMID": pmid,
            "Authors": authors,
        })

    return records

# ========= SCHOLAR PARSER =========
def parse_scholar(text):
    records = []
    entries = re.split(r'\[\d+\]', text)

    for entry in entries:
        if "Title:" not in entry:
            continue

        title_match = re.search(r'Title:\s*(.+)', entry)
        title = title_match.group(1).strip() if title_match else ""
        authors_match = re.search(r'Authors:\s*(.+)', entry)
        authors = authors_match.group(1) if authors_match else ""

        records.append({
            "Database": "Google Scholar",
            "Title": title,
            "DOI": "",
            "PMID": "",
            "Authors": authors,
        })

    return records

# ========= LECTURE FICHIERS =========

with open("IEEE Xplore Citation Plain Text Download 2026.3.31.16.1.40.txt", encoding="utf-8") as f:
    ieee_text = f.read()

with open("scopus_export_Mar 31-2026_fc3be87f-2e9d-4ea0-8c22-2e3d855f3cec.txt", encoding="utf-8") as f:
    scopus_text = f.read()

with open("google_scholar.txt", encoding="utf-8") as f:
    scholar_text = f.read()

# ========= PARSING =========

records = []
records += parse_ieee(ieee_text)
records += parse_scopus(scopus_text)
records += parse_scholar(scholar_text)

df = pd.DataFrame(records)

# ========= NORMALISATION =========

df["Title_norm"] = df["Title"].apply(normalize)
df["DOI_norm"] = df["DOI"].apply(normalize)

# ========= DUPLICATES EXACT =========

df["IsDuplicate"] = df.duplicated(subset=["DOI_norm", "Title_norm"], keep=False)

# ========= FUZZY MATCHING =========

threshold = 85  # 🔥 abaissé pour mieux détecter

fuzzy_duplicates = []
similarity_scores = []

for i in range(len(df)):
    is_dup = False
    max_score = 0

    for j in range(len(df)):
        if i == j:
            continue

        # priorité au DOI
        if df.loc[i, "DOI_norm"] and df.loc[i, "DOI_norm"] == df.loc[j, "DOI_norm"]:
            is_dup = True
            max_score = 100
            break

        # comparaison titres
        score = fuzz.ratio(df.loc[i, "Title_norm"], df.loc[j, "Title_norm"])

        if score > max_score:
            max_score = score

        if score >= threshold:
            is_dup = True

    fuzzy_duplicates.append(is_dup)
    similarity_scores.append(max_score)

df["IsDuplicate_fuzzy"] = fuzzy_duplicates
df["SimilarityScore"] = similarity_scores

# ========= STATS =========

counts = df["Database"].value_counts().reset_index()
counts.columns = ["Database", "Count"]
total_count = len(df)

print("\nNombre d'articles par base :")
print(counts)
print(f"\nTotal d'articles : {total_count}")

# ========= EXPORT =========

with pd.ExcelWriter("All_articles_withAuthors.xlsx") as writer:
    df.to_excel(writer, sheet_name="Records", index=False)
    counts.to_excel(writer, sheet_name="Counts", index=False)

print("Excel généré : All_articles_withAuthors.xlsx")


Nombre d'articles par base :
         Database  Count
0          Scopus    187
1  Google Scholar     30
2            IEEE     25

Total d'articles : 242
Excel généré : All_articles_withAuthors.xlsx
